# 02. 저장된 raw 데이터 전처리
인터넷/API 호출 없이 반복 실행할 수 있습니다.

In [1]:
from pathlib import Path
import sys
import pandas as pd
PROJECT_ROOT = Path('/Users/choedasom/lab_middle_project')
assert (PROJECT_ROOT / 'src' / 'preprocess.py').is_file(), f'경로 확인 필요: {PROJECT_ROOT}'
if str(PROJECT_ROOT) not in sys.path: sys.path.insert(0, str(PROJECT_ROOT))

for m in list(sys.modules):
    if m == 'src' or m.startswith('src.'):
        del sys.modules[m]

from src.preprocess import build_ml_dataset
RAW_DIR = PROJECT_ROOT / 'data' / 'raw'
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

In [2]:
final_df_raw = pd.read_parquet(RAW_DIR / 'final_df_raw.parquet')
collection_missing_df = pd.read_csv(RAW_DIR / 'final_missing_df.csv')
sp500_universe = pd.read_csv(RAW_DIR / 'sp500_universe.csv')
print(final_df_raw.shape, collection_missing_df.shape, sp500_universe.shape)

(1663038, 8) (21, 7) (729, 4)


In [ ]:
# --- 알려진 데이터 오염 종목 제외 (2026-08-29 발견, 2026-09-01 검증 보강) ---
# PARA: raw 가격 시계열의 상당 구간이 주식이 아니라 채권 가격으로 추정되는 값으로 오염됨
#   (액면가 100 근처 저변동성, 거래량 한 자릿수대, 가격이 정수×1000 형태로 저장
#    — 회사채 가격 시스템의 전형적 특징). 발행사 식별자 충돌 등으로 채권 시계열이
#    잘못 병합된 것으로 추정. 오염 경계가 불명확해 부분 복구 대신 종목 전체 제외.
#    다운스트림 영향(제외 전 측정): rebalance_60df 안정그룹 스냅샷 기준 18/12,806행(0.14%).
#
# 오염 지점 = 우리 코드가 아니라 API 응답 자체. final_df_raw에서 PARA 1,350행의
# source가 전부 'yahoo'(fetch_yahoo, yf.Ticker().history() 직접 호출 경로)였고,
# 그 경로의 _tidy()는 pd.to_numeric() 타입 변환만 할 뿐 배율/스케일 조정 로직이
# 전혀 없음(분할/배당 조정 비율을 곱하는 코드는 fetch_yahoo_chart 경로에만 있는데,
# PARA는 그 경로를 안 탐 — source가 'yahoo:chart'가 아니라 'yahoo'인 것으로 확인).
# 즉 우리 파이프라인이 값을 바꾼 게 아니라, Yahoo Finance 서버가 애초에 오염된
# 값을 응답으로 준 것을 그대로 저장한 것.
#
# 검증 과정 (2026-09-01):
#   1. 월별로 뜯어보니 2021-02~2023-11 구간은 96,000~107,000 사이에서 거의 안
#      움직이고 거래량도 0~30주 — 회사채가 액면가 근처에서 조용히 거래되는
#      패턴과 정확히 일치.
#   2. 2023-12부터 값이 붕괴하며(같은 달 안에 18,800~113,900 혼재) 이후
#      2026-06-30까지 25,700 -> 2.78로 부드럽게 우하향하는 별개의 이상 패턴이
#      이어짐 — "2021~2023만 오염"이라는 최초 판단보다 오염 범위가 넓음.
#   3. 결정적 증거: Paramount는 2025-08-07 Skydance와 합병 완료되며 티커가
#      PARA -> PSKY로 바뀌었는데(실제 뉴스로 확인), raw 데이터엔 2026-06-30까지
#      PARA 행이 남아있음 — 이미 존재하지 않는 티커로 거래가 찍히는 것 자체가
#      전체 시계열이 신뢰 불가하다는 뜻. 종목 전체 제외 결정이 오히려 더 타당함.
EXCLUDED_TICKERS = ['PARA']

n_before = len(final_df_raw)
final_df_raw = final_df_raw[~final_df_raw['Ticker'].isin(EXCLUDED_TICKERS)].reset_index(drop=True)
print(f'제외 적용: {n_before - len(final_df_raw):,}행 제거 ({EXCLUDED_TICKERS})')

# 감사 추적: final_missing_df에도 기록 (수집 실패 21종목과는 fail_stage로 구분)
excluded_record = pd.DataFrame({
    'ticker': EXCLUDED_TICKERS,
    'company': sp500_universe.set_index('ticker')['company'].reindex(EXCLUDED_TICKERS).fillna('Unknown').values,
    'sector': sp500_universe.set_index('ticker')['sector'].reindex(EXCLUDED_TICKERS).fillna('Unknown').values,
    'fail_stage': 'data_quality',
    'fail_reason': '채권 가격 오염 추정 — 부분 구간 액면가 근접 저변동성 패턴, 상세는 위 주석',
})
collection_missing_df = pd.concat([collection_missing_df, excluded_record], ignore_index=True)

# 후속 조사 (2026-09-02): PARA 말고 비슷하게 오염된 종목이 더 있는가

PARA를 찾을 때 쓴 기준(`price_range_ratio` = 종목별 `max(Close)/min(Close)`, 기존
가드가 "1000 넘으면 분할 미조정 의심"이라고 이미 표시해주던 값)을 PARA 하나에만
쓰지 않고 **전체 티커에 돌려서** 비슷한 종목이 더 있는지 확인했다.

**결과**: PARA 제외하고 `price_range_ratio > 1000`인 종목 2개 — `CHK`(1,059),
`SIVB`(580,792).

## CHK — 정상, 조치 불필요
2020년 파산 → 2021년 1:200 리버스 스플릿 → 2024-10 Expand Energy(EXE)로 합병되며
상장폐지. 데이터의 최고가(2016년)/최저가(2020년 말)/종료 시점(2024-10-04) 전부
실제 역사와 정확히 일치. 기존 단일거래일 가드(스플릿 실행일의 급등)가 이미
처리하고 있음.

## SIVB — 처음엔 PARA처럼 의심됐지만, 조사해보니 제외하면 안 됨
`SIVB`(실리콘밸리은행)는 2023-03-09 실제 파산 붕괴 이후 데이터가 다음처럼 이어짐:
- 2023-03-10~03-20: $106.04 고정, 거래량 0 (거래정지 상태를 그대로 반영 — 정상)
- 2023-11~2024-11: $0.006까지 서서히 하락 (파산 절차 중 OTC 페니주 거래로 추정)
- **2024-11~2026-02: 약 14개월치 데이터가 통째로 빔**
- **2026-02~2026-06: $0.0060으로 5개월 내내 소수점까지 완전히 고정** — 진짜 시장
  거래라면 나올 수 없는 패턴 → PARA와 같은 "존재하지 않아야 할 시점까지 데이터가
  남아있는" 빨간 깃발로 보고 추가 조사함.

**조사 결과 — PARA와 근본적으로 다른 케이스라고 판단, 제외하지 않기로 결정**:
1. `_load_membership_history()` 기준 SIVB의 실제 S&P500 편입 종료일이
   `2023-03-15`로, 실제 파산·편출 시점과 정확히 일치.
2. `filter_by_membership()`이 이미 이 날짜 이후 데이터를 자동 차단 —
   `rebalance_60df`/`rebalance_120df`의 SIVB 마지막 행은 `2023-02-28`(파산 *전*)이고,
   문제였던 2024-11 이후 구간은 애초에 후보에 들어간 적이 없음.
3. `add_forward_returns()`가 이 마지막 진입일의 청산가를 조회하는 시점도
   60일 기준 ~2023-04, 120일 기준 `2023-08-21`(실거래량 43,569주 있음)로,
   "진짜 붕괴 직후의 실거래" 구간까지만 닿고 문제의 2024-11 이후 구간까지는
   안 닿음.

**PARA와의 핵심 차이**: PARA는 실제 S&P500 멤버였던 기간(2021~2023) *한복판*에
가짜 데이터(무관한 채권 가격)가 섞여 있어서 `filter_by_membership()`으로도 못
걸렀다. SIVB는 오염이 실제 편출 *이후*에만 있고, 편출 시점 자체가 실제 파산
시점과 정확히 일치해서 point-in-time 필터가 이미 완벽하게 막아준다. 게다가
SIVB의 (필터를 통과하는) 하락 구간은 가짜 데이터가 아니라 **진짜 파산으로 인한
실제 손실**이라, 이걸 지우면 "안정성 우선 전략이 실제 파산 리스크를 어떻게
다뤘는지"를 보여주는 유효한 신호까지 같이 지워버리는 셈이 된다. 그래서 SIVB는
그대로 둔다.

In [4]:
final_df, coverage_df, final_missing_df = build_ml_dataset(
    final_df_raw, universe=sp500_universe, final_missing_df=collection_missing_df
)
display(final_df.head())
display(coverage_df.head())
display(final_missing_df.head())

,Date,Ticker,Open,High,Low,Close,Volume,source
0,2016-01-04,A,37.751924,37.871448,37.089932,37.411732,3287300,yahoo
1,2016-01-05,A,37.448518,37.650795,37.089940,37.283020,2587200,yahoo
2,2016-01-06,A,36.997993,37.687568,36.823298,37.448513,2103600,yahoo
3,2016-01-07,A,36.906036,36.915233,35.683192,35.857883,3504300,yahoo
4,2016-01-08,A,36.060163,36.510683,35.370588,35.480919,3736700,yahoo


,Ticker,n_rows,actual_start_date,actual_end_date,n_price_imputed,n_ohlc_inconsistent,n_volume_missing,sources,expected_rows_10y,coverage_10y,short_history,has_quality_issue,company,sector
0,A,2637,2016-01-04,2026-06-30,0,0,0,yahoo,2738,0.9631,False,False,Agilent Technologies,Health Care
1,AABA,948,2016-01-04,2019-11-06,0,0,0,tiingo,2738,0.3462,True,False,AABA,Unknown
2,AAL,2637,2016-01-04,2026-06-30,0,0,0,yahoo,2738,0.9631,False,False,AAL,Unknown
3,AAP,2637,2016-01-04,2026-06-30,0,0,0,yahoo,2738,0.9631,False,False,AAP,Unknown
4,AAPL,2637,2016-01-04,2026-06-30,0,0,0,yahoo,2738,0.9631,False,False,Apple Inc.,Information Technology


,Ticker,company,sector,fail_stage,fail_reason,n_rows_raw,n_rows_final
0,ADS,ADS,Unknown,collection,yahoo: empty response | chart: HTTP 404 | tiin...,0,0
1,BBBY,BBBY,Unknown,collection,yahoo: empty response | chart: HTTP 400 | tiin...,0,0
2,BK,BK,Unknown,collection,yahoo: empty response | chart: HTTP 404,0,0
3,CBS,CBS,Unknown,collection,yahoo: empty response | chart: HTTP 404 | tiin...,0,0
4,CCE,CCE,Unknown,collection,yahoo: empty response | chart: no data | tiing...,0,0


In [5]:
final_df.to_parquet(PROCESSED_DIR / 'final_df.parquet', index=False)
coverage_df.to_csv(PROCESSED_DIR / 'coverage_df.csv', index=False)
final_missing_df.to_csv(PROCESSED_DIR / 'final_missing_df.csv', index=False)
print(f'processed 저장 완료: {PROCESSED_DIR}')

processed 저장 완료: /Users/choedasom/lab_middle_project/data/processed
